#Objective
Employee Data Analysis using PySpark (GroupBy & Aggregations)

Step 1: Importing library

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg
from pyspark.sql.functions import max
from pyspark.sql.functions import count
from pyspark.sql.functions import desc
from pyspark.sql.functions import asc
import pyspark.sql.functions as F
spark=SparkSession.builder.getOrCreate()


Step 2: Creating Hardcoded Data
Since using a CSV file from my local system can cause accessibility issues (for example, others won’t have access to the same file path), I am using hardcoded data in this step to ensure the notebook runs successfully for everyone.

In the upcoming steps, I will switch to loading the dataset from a GitHub link so that the notebook becomes fully reproducible and easy for others (like my mentor) to review.

In [ ]:

data = [
    ("Krish","Data Science",10000),
    ("Krish","IOT",5000),
    ("Mahesh","Big Data",4000),
    ("Krish","Big Data",4000),
    ("Mahesh","Data Science",3000),
    ("Sudhanshu","Data Science",20000),
    ("Sudhanshu","IOT",10000),
    ("Sudhanshu","Big Data",5000),
    ("Sunny","Data Science",10000),
    ("Sunny","Big Data",2000)
]

columns=["Name","Department","Salary"]
df=spark.createDataFrame(data,columns)
df.show()

+---------+------------+------+
|     Name|  Department|Salary|
+---------+------------+------+
|    Krish|Data Science| 10000|
|    Krish|         IOT|  5000|
|   Mahesh|    Big Data|  4000|
|    Krish|    Big Data|  4000|
|   Mahesh|Data Science|  3000|
|Sudhanshu|Data Science| 20000|
|Sudhanshu|         IOT| 10000|
|Sudhanshu|    Big Data|  5000|
|    Sunny|Data Science| 10000|
|    Sunny|    Big Data|  2000|
+---------+------------+------+



#Now After importing libraries and data now u can do analysis

Just grouping the data as per need


In [ ]:
df.groupBy("Department").count().show()
#we can directly write like this
#df.groupBy("Department").show() it shows an error


#groupBy() does NOT return a DataFrame
#It returns a GroupedData object
# And .show() works only on DataFrames

+------------+-----+
|  Department|count|
+------------+-----+
|         IOT|    2|
|    Big Data|    4|
|Data Science|    4|
+------------+-----+



In [ ]:
#Doing this to find department wise salary

In [ ]:
df.groupBy("Department") \
  .agg(avg("Salary").alias("Average_Salary")) \
  .show()
#.groupby is used to make common things as a group
#.agg is used Because you can do multiple operations at once
# avg is used to find avg of numerical column
#.alias is used to give a readable name to column

+------------+--------------+
|  Department|Average_Salary|
+------------+--------------+
|         IOT|        7500.0|
|    Big Data|        3750.0|
|Data Science|       10750.0|
+------------+--------------+



#Using max and group by to see department wise max salary

In [ ]:
df.groupBy("Department").max("Salary").show()

+------------+-----------+
|  Department|max(Salary)|
+------------+-----------+
|         IOT|      10000|
|    Big Data|       5000|
|Data Science|      20000|
+------------+-----------+



#So we use orderBy and desc function to see which employe having max salary

In [ ]:
df.orderBy(df["Salary"].desc()).show(1)
#orderBy is used to arrange column in a order manager we use desc here
#so our data is arranged in desc order and we fetch 1st row using show(1)
#By default it show gives 20 rows

+---------+------------+------+
|     Name|  Department|Salary|
+---------+------------+------+
|Sudhanshu|Data Science| 20000|
+---------+------------+------+
only showing top 1 row


In [ ]:
df.groupBy("Department").agg(count("*").alias("Working_employee_per_department")).show()
#groupby to make group of common departments
#.agg to use multiple opeartions at once
#Count("*") to read all rows in department
#.alias give readable name to column

+------------+-------------------------------+
|  Department|Working_employee_per_department|
+------------+-------------------------------+
|         IOT|                              2|
|    Big Data|                              4|
|Data Science|                              4|
+------------+-------------------------------+



#Just checking schema

In [ ]:
df.printSchema()

root
 |-- Name: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- Salary: long (nullable = true)



#Showing salary after bonus

In [ ]:
df.withColumn("After_bonuc_salary",df['Salary']+2000).show(5)

+------+------------+------+------------------+
|  Name|  Department|Salary|After_bonuc_salary|
+------+------------+------+------------------+
| Krish|Data Science| 10000|             12000|
| Krish|         IOT|  5000|              7000|
|Mahesh|    Big Data|  4000|              6000|
| Krish|    Big Data|  4000|              6000|
|Mahesh|Data Science|  3000|              5000|
+------+------------+------+------------------+
only showing top 5 rows


#Showing minimum salary employe details

In [ ]:
df.orderBy(df["Salary"].asc()).show(1)

+-----+----------+------+
| Name|Department|Salary|
+-----+----------+------+
|Sunny|  Big Data|  2000|
+-----+----------+------+
only showing top 1 row


#Adding a power block which can perform multiple opeartions at once instead of doing multiple single opeartions

In [ ]:
df.groupBy("Department").agg(F.count("*").alias("employe_per_department"),F.avg("Salary").alias("avg_salary_per_department"),F.max("Salary").alias("Max_salary")).show()

+------------+----------------------+-------------------------+----------+
|  Department|employe_per_department|avg_salary_per_department|Max_salary|
+------------+----------------------+-------------------------+----------+
|         IOT|                     2|                   7500.0|     10000|
|    Big Data|                     4|                   3750.0|      5000|
|Data Science|                     4|                  10750.0|     20000|
+------------+----------------------+-------------------------+----------+



#Filtering High earners who has more than 8000 salary

In [ ]:
df.filter(df["Salary"]>=8000).show()
#i tried alias here but alias only work for columns not for filtering dataframes

+---------+------------+------+
|     Name|  Department|Salary|
+---------+------------+------+
|    Krish|Data Science| 10000|
|Sudhanshu|Data Science| 20000|
|Sudhanshu|         IOT| 10000|
|    Sunny|Data Science| 10000|
+---------+------------+------+



#Show all employees who belong to Data Science department

In [ ]:
df.filter(df["Department"]=="Data Science").show()

+---------+------------+------+
|     Name|  Department|Salary|
+---------+------------+------+
|    Krish|Data Science| 10000|
|   Mahesh|Data Science|  3000|
|Sudhanshu|Data Science| 20000|
|    Sunny|Data Science| 10000|
+---------+------------+------+



#Get employees:

Salary > 5000
AND Department = "Data Science"

In [ ]:
df.filter((df["Department"]=="Data Science") & (df["Salary"]>5000)).show()

+---------+------------+------+
|     Name|  Department|Salary|
+---------+------------+------+
|    Krish|Data Science| 10000|
|Sudhanshu|Data Science| 20000|
|    Sunny|Data Science| 10000|
+---------+------------+------+



#Get employees:

Salary > 15000
OR Department = "IOT"

In [ ]:
df.filter((df["Department"]=="IOT") | (df["Salary"]>15000)).show()

+---------+------------+------+
|     Name|  Department|Salary|
+---------+------------+------+
|    Krish|         IOT|  5000|
|Sudhanshu|Data Science| 20000|
|Sudhanshu|         IOT| 10000|
+---------+------------+------+



#Get employees who are NOT in IOT

In [ ]:
df.filter(df['Department']!="IOT").show()

+---------+------------+------+
|     Name|  Department|Salary|
+---------+------------+------+
|    Krish|Data Science| 10000|
|   Mahesh|    Big Data|  4000|
|    Krish|    Big Data|  4000|
|   Mahesh|Data Science|  3000|
|Sudhanshu|Data Science| 20000|
|Sudhanshu|    Big Data|  5000|
|    Sunny|Data Science| 10000|
|    Sunny|    Big Data|  2000|
+---------+------------+------+



#Get employees:

Salary between 5000 and 15000

In [ ]:
#df.filter((df["Salary"] >= 5000) & (df["Salary"] <= 15000)).show()

#using built in method between
df.filter((df["Salary"].between(5000,15000))).show()

+---------+------------+------+
|     Name|  Department|Salary|
+---------+------------+------+
|    Krish|Data Science| 10000|
|    Krish|         IOT|  5000|
|Sudhanshu|         IOT| 10000|
|Sudhanshu|    Big Data|  5000|
|    Sunny|Data Science| 10000|
+---------+------------+------+



#Printing 2nd highest salary

In [ ]:
df.orderBy(df["Salary"].desc()).collect()[1]
#we use .collect to use 1st index value we can use different methods to print second highest salary

Row(Name='Krish', Department='Data Science', Salary=10000)

#Checking for distinct domains
select() → choose columns

filter() → choose rows

In [ ]:
df.select("Department").distinct().show()

+------------+
|  Department|
+------------+
|         IOT|
|    Big Data|
|Data Science|
+------------+



#Just trying to dlt a column

In [ ]:
df.drop("Department").show()

+---------+------+
|     Name|Salary|
+---------+------+
|    Krish| 10000|
|    Krish|  5000|
|   Mahesh|  4000|
|    Krish|  4000|
|   Mahesh|  3000|
|Sudhanshu| 20000|
|Sudhanshu| 10000|
|Sudhanshu|  5000|
|    Sunny| 10000|
|    Sunny|  2000|
+---------+------+



#Conclusion

1.Worked on employee data analysis using PySpark basics

2.Used hardcoded dataset so the notebook runs easily for everyone

3.Learned DataFrame operations like groupBy, agg, filter, and orderBy

4.Calculated average salary, max salary, and employee count per department

5.Used sorting to find highest and lowest salary employees
Applied filtering with different conditions (AND, OR, NOT, between)

6.Added new column using withColumn (bonus salary)

7.Checked schema to understand data types

8.Used distinct and drop for better data handling

9.Learned how to find second highest salary

10.Combined multiple operations in one query

Overall, got a good understanding of PySpark basics and data handling
Next, I will learn joins, window functions, and ML pipeline